# Analyse, classification et indexation des données: feuille 9
### Apprentissage supervisé, Arbres de décision et Forêts aléatoires

## Avant propos

Dans cette feuille, nous allons étudier une catégorie d'algorithmes d'apprentissage supervisé différente de ce que nous avons vu jusque là : 

- les arbres de décisions ont cela de particulier qu'ils sont construits en utilisant des éléments de la théorie de l'information. Leur utilisation, ensuite, est assez facilement implémentable par des règles Si-Sinon simples. 

- les forêts aléatoires implémentent, elles, une approche de bagging.

In [1]:
import numpy as np
import pandas as pa
import warnings
warnings.filterwarnings("ignore")

### Exercice 1. Arbres de décision (à la main)

Dans ce premier exercice, nous allons écrire un ensemble de fonctions implémentant les éléments vus en cours et permettant de construire un arbre de décision.  

Pour vérifier vos fonctions, vous pouvez les appliquer à l'exemple très simple vu en cours :


In [2]:
exemple_cours = pa.DataFrame(data={'A': [0, 0, 1, 1],
                                  'B': [1, 0, 1, 0],
                                  'Classe': ['C1', 'C1', 'C2', 'C2']})

1- Ecrire une fonction <code>information(dataset, label_feature)</code> permettant de calculer la quantité d'information nécessaire pour classifier un élément du corpus <code>dataset</code>. Le paramètre <code>label_feature</code> indique la colonne contenant les classes des éléments.

In [ ]:
def information(dataset, label_feature)


2- Ecrire une fonction <code>entropy(dataset, feature, label_feature)</code> calculant l'entropie du descripteur <code>feature</code>

3- Ecrire une fonction <code>gain(dataset, feature, label_feature)</code> calculant le gain d'information obtenu en choisissant le descripteur <code>feature</code> pour partitionner le corpus <code>dataset</code>

4- En vous aidant des fonctions que vous avez écrites ci-dessus, construisez l'arbre de décision pour prédire Achat avec le corpus suivant. On ne demande pas d'écrire un programme qui construit l'arbre mais plutôt d'utiliser les fonctions pour faire les calculs à votre place.

\begin{array}{|c|c|c|c|c|}
\hline
\hline
Sexe & Age & Etat civil & Revenu & Achat \\
\hline
Homme & 18-35   & Marie       & Moyen  & Non \\
Homme & <18     & Celibataire & Faible & Non \\
Homme & >35     & Marie       & Eleve  & Oui \\
Femme & <18     & Celibataire & Moyen  & Non \\
Homme & 18 - 35 & Celibataire & Moyen  & Non \\
Femme & 18 - 35 & Celibataire & Eleve  & Oui \\
Femme & 18 - 35 & Marie       & Faible & Non \\
Homme & 18 - 35 & Marie       & Eleve  & Oui \\
Homme & >35     & Celibataire & Faible & Oui \\
Femme & <18     & Celibataire & Moyen  & Non \\
Femme & >35     & Celibataire & Moyen  & Oui \\
Femme & >35     & Marie       & Eleve  & Oui \\
Homme & 18 - 35 & Celibataire & Faible & Non\\
Femme & 18 - 35 & Marie       & Moyen  & Oui\\
\hline
\hline
\end{array}

In [ ]:
dataset = pa.DataFrame(data={'Sexe': ['Homme', 'Homme', 'Homme', 'Femme','Homme', 'Femme', 'Femme', 
                                'Homme', 'Homme', 'Femme', 'Femme', 'Femme','Homme', 'Femme'],
                      'Age': ['18-35', '<18', '>35', '<18', '18-35', '18-35', '18-35', '18-35', 
                              '>35', '<18', '>35', '>35', '18-35', '18-35'],
                      'Etat_civil': ['marie', 'celibataire', 'marie', 'celibataire', 'celibataire', 
                                     'celibataire', 'marie', 'marie', 'celibataire', 'celibataire',
                                    'celibataire', 'marie', 'celibataire', 'marie'],
                      'Revenu': ['Moyen', 'Faible', 'Eleve', 'Moyen', 'Moyen', 'Eleve', 'Faible',
                                'Eleve', 'Faible', 'Moyen', 'Moyen', 'Eleve', 'Faible', 'Moyen'],
                      'Achat': ['Non', 'Non', 'Oui', 'Non', 'Non', 'Oui', 'Non', 'Oui', 'Oui', 'Non',
                               'Oui', 'Oui', 'Non', 'Oui']})
#dataset

5- Dessiner l'arbre de décision obtenu.

## Exercice 2. Arbres de décision avec <code>sklearn</code>

Dans cet exercice, nous allons utiliser la bibliothèque <code>sklearn</code> pour entraîner et tester un arbre de décision. 

Pour cela, nous allons utiliser le corpus <code>titanic.csv</code>. Un échantillon de ce corpus est diponible à l'adresse : 

https://www.labri.fr/perso/zemmari/datasets/titanic.csv

Une description détaillée de ce corpus et des descripteurs le composant peut être consultée à l'adresse :

https://www.kaggle.com/c/titanic/data

On vous laisse découvrir la tragédie à laquelle est lié ce corpus :-)


1- Chargez le corpus et explorez son contenu.

In [3]:
dt = pa.read_csv('titanic.csv')

In [6]:
dt.head()
df = dt.dropna()
df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
6,7,0,1,"McCarthy, Mr. Timothy J",male,54.0,0,0,17463,51.8625,E46,S
10,11,1,3,"Sandstrom, Miss. Marguerite Rut",female,4.0,1,1,PP 9549,16.7000,G6,S
11,12,1,1,"Bonnell, Miss. Elizabeth",female,58.0,0,0,113783,26.5500,C103,S


2- Supprimez les lignes contenant des données manquantes

3- On souhaite construire un modèle pour prédire, pour un passager, s'il survivra ou non. Supprimez les descripteurs qui ne vous paraissent pas pertinents pour la prédiction. Expliquez vos choix.

In [18]:
dataset = df.drop(["PassengerId","Name","Ticket","Cabin"],axis=1)

4- La cellule suivante permet de convertir une catégorie au format entier ou au format texte en un entier. 
Exécutez la telle quelle et observez son résultat.

In [19]:
from sklearn import preprocessing

cols = ['Sex', 'Embarked']
le = preprocessing.LabelEncoder()
for c in cols:
    le.fit(dataset[c])
    dataset[c] = le.transform(dataset[c])
dataset

,Survived,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
1,1,1,0,38.0,1,0,71.2833,0
3,1,1,0,35.0,1,0,53.1000,2
6,0,1,1,54.0,0,0,51.8625,2
10,1,3,0,4.0,1,1,16.7000,2
11,1,1,0,58.0,0,0,26.5500,2
...,...,...,...,...,...,...,...,...
871,1,1,0,47.0,1,1,52.5542,2
872,0,1,1,33.0,0,0,5.0000,2
879,1,1,0,56.0,0,1,83.1583,0
887,1,1,0,19.0,0,0,30.0000,2


5- Séparez les données en un ensemble $X$ contenant les variables explicatives et $y$ la variable à prédire.

In [27]:
X = dataset.drop(["Survived"], axis=1)
y = dataset['Survived']

6- Partitionnez les données en ensemble d'entraînement et de test

In [28]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

7- Instanciez un classifieur Decision Tree et entraînez-le avec les données d'entraînement.

In [43]:
from sklearn.tree import DecisionTreeClassifier
dt = DecisionTreeClassifier(max_depth=2, min_samples_leaf=5)
dt.fit(X_train, y_train)

,criterion,'gini'
,splitter,'best'
,max_depth,2
,min_samples_split,2
,min_samples_leaf,5
,min_weight_fraction_leaf,0.0
,max_features,None
,random_state,None
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,class_weight,None


8- Visualisez l'arbre obtenu. Décommentez/complétez pour cela le code suivant

In [44]:
from sklearn.tree import export_graphviz
from six import StringIO  
from IPython.display import Image
import pydotplus

features_cols = ['Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare', 'Embarked']
class_names=['0','1']

dot_data = StringIO()
export_graphviz(dt, out_file=dot_data,  
                filled=True, rounded=True,
               special_characters=True,feature_names = features_cols,class_names=class_names)
graph = pydotplus.graph_from_dot_data(dot_data.getvalue())  
graph.write_png('titanic.png')
Image(graph.create_png())

InvocationException: GraphViz's executables not found

9- Mesurez la qualité de votre classifieur.

In [45]:
from sklearn.metrics import accuracy_score
y_pred = dt.predict(X_test)
print("Précision:", accuracy_score(y_test, y_pred))

Précision: 0.7837837837837838


10- Changez les valeurs par défaut des différents paramètres et observez la qualité de votre classifieur. 

## Exercice 3. Random forest

Quelles sont les performances d'un classifieur de type forêts aléatoires sur le même corpus que l'exercice précédant ?



Cherchez le meilleur paramètrage pour le meilleur classifieur en complétant / décommentant le code suivant.

In [49]:
from sklearn.model_selection import RandomizedSearchCV
import numpy as np

# --- 1. Création de la grille de paramètres à tester ---

# Nombre d'arbres dans la forêt (de 200 à 2000)
n_estimators = [int(x) for x in np.linspace(start = 200, stop = 2000, num = 10)]

# Nombre de features à considérer à chaque split
# 'sqrt' est la méthode classique (racine carrée du nombre total de features)
max_features = ['sqrt', 'log2'] 

# Profondeur maximale de l'arbre
max_depth = [int(x) for x in np.linspace(10, 110, num = 11)]
max_depth.append(None) # None signifie pas de limite

# Nombre minimum d'échantillons requis pour diviser un nœud
min_samples_split = [2, 5, 10]

# Nombre minimum d'échantillons requis à chaque nœud feuille
min_samples_leaf = [1, 2, 4]

# Méthode de sélection des échantillons pour entraîner chaque arbre
# True = avec remise (bootstrap), False = tout le dataset
bootstrap = [True, False]

# Création du dictionnaire de la grille aléatoire
random_grid = {'n_estimators': n_estimators,
               'max_features': max_features,
               'max_depth': max_depth,
               'min_samples_split': min_samples_split,
               'min_samples_leaf': min_samples_leaf,
               'bootstrap': bootstrap}

print("Grille de paramètres définie :")
print(random_grid)

Grille de paramètres définie :
{'n_estimators': [200, 400, 600, 800, 1000, 1200, 1400, 1600, 1800, 2000], 'max_features': ['sqrt', 'log2'], 'max_depth': [10, 20, 30, 40, 50, 60, 70, 80, 90, 100, 110, None], 'min_samples_split': [2, 5, 10], 'min_samples_leaf': [1, 2, 4], 'bootstrap': [True, False]}


In [51]:
from sklearn.ensemble import RandomForestClassifier 
# Use the random grid to search for best hyperparameters
# First create the base model to tune
rf = RandomForestClassifier()
# Random search of parameters, using 3 fold cross validation, 
# search across 100 different combinations, and use all available cores
rf_random = RandomizedSearchCV(estimator = rf, param_distributions = random_grid, n_iter = 100, 
                               cv = 3, verbose=2, random_state=42, n_jobs = -1)
rf_random.fit(X_train, y_train)

Fitting 3 folds for each of 100 candidates, totalling 300 fits


,estimator,RandomForestClassifier()
,param_distributions,"{'bootstrap': [True, False], 'max_depth': [10, 20, ...], 'max_features': ['sqrt', 'log2'], 'min_samples_leaf': [1, 2, ...], ...}"
,n_iter,100
,scoring,None
,n_jobs,-1
,refit,True
,cv,3
,verbose,2
,pre_dispatch,'2*n_jobs'
,random_state,42
,error_score,nan


In [52]:
rf_random.best_params_

{'n_estimators': 400,
 'min_samples_split': 2,
 'min_samples_leaf': 1,
 'max_features': 'log2',
 'max_depth': None,
 'bootstrap': False}

In [53]:
# --- 3. Résultats ---

print("\nMeilleurs paramètres trouvés :")
print(rf_random.best_params_)

best_random_model = rf_random.best_estimator_
best_acc = accuracy_score(y_test, best_random_model.predict(X_test))

print(f"\nPrécision du meilleur modèle après optimisation : {best_acc:.4f}")


Meilleurs paramètres trouvés :
{'n_estimators': 400, 'min_samples_split': 2, 'min_samples_leaf': 1, 'max_features': 'log2', 'max_depth': None, 'bootstrap': False}

Précision du meilleur modèle après optimisation : 0.7297
